# 16 — Synthèse finale et choix du modèle`

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

A = C.DATA / "09_Resultats_MembreA" / "livrables_membre_A"
CORR = A / "2_version_corrigee"
INIT = A / "1_version_initiale_protocole_fiche"

def dernier(motif):
    d = sorted(C.COHORTE.glob(motif))
    return d[-1] if d else None

sources = {
    "Membre A — corrigee": CORR if CORR.is_dir() else None,
    "Membre A — initiale": INIT if INIT.is_dir() else None,
    "Membre A — stabilite CNN": (A / "3_stabilite_CNN") if (A / "3_stabilite_CNN").is_dir() else None,
    "B — 11b modeles (ancree)": dernier("comparaison_modeles_ancree_*"),
    "B — 11 modeles (initiale)": dernier("comparaison_modeles_[0-9]*"),
    "B — 12 longitudinal": dernier("longitudinal_RQ2_RQ3_*"),
    "B — 13 Spark": dernier("pipeline_spark_*"),
    "B — 07c cible ancree": dernier("cible_ancree_359_*"),
}
for nom, chemin in sources.items():
    print(f"{nom:28s} : {chemin.name if chemin else 'ABSENT'}")

In [ ]:
# Effectifs des cohortes (pour annoter chaque ligne du tableau).
N = {"A_corr": 85, "A_init": 112}
d = sources["B — 07c cible ancree"]
if d is not None:
    j = pd.read_csv(d / "jeu_modelisation_359_ancre.csv")
    arisque = j.loc[(j["deja_dement"] == 0) & j["progression_ancree"].notna()]
    N["B_ancree"] = len(arisque)
    print("Cohorte 11b (359 ancree, a risque) :", N["B_ancree"],
          "| progressent :", int(arisque["progression_ancree"].sum()))
else:
    N["B_ancree"] = None

lignes = []   # tableau de synthese, une ligne par resultat

def ajouter(rq, question, modele, cohorte, n, auc, ect, notebook):
    lignes.append({"RQ": rq, "Question": question, "Modele / configuration": modele,
                   "Cohorte": cohorte, "N": n, "ROC-AUC": round(float(auc), 3),
                   "ecart-type": None if ect is None else round(float(ect), 3),
                   "Notebook": notebook})

def nombre(x):
    """Lit '0.711 ± 0.060' ou 0.711."""
    if isinstance(x, str) and "±" in x:
        a, b = x.split("±")
        return float(a.strip()), float(b.strip())
    return float(x), None

In [ ]:
# ---------- RQ1 : la fusion multimodale apporte-t-elle quelque chose ? ----------
if sources["Membre A — corrigee"]:
    t = pd.read_csv(CORR / "RQ1_comparaison.csv", index_col=0)
    for modele in t.index:
        ajouter("RQ1", "Fusion vs unimodal", modele, "imagerie, a risque",
                N["A_corr"], t.loc[modele, "roc_auc"],
                t.loc[modele].get("roc_auc_ecart_type"), "10")

    cnn = pd.read_csv(CORR / "resultats_CNN.csv", index_col=0)
    for conf in cnn.index:
        m, s = nombre(cnn.loc[conf, "ROC-AUC"])
        ajouter("RQ1", "Fusion vs unimodal (deep learning)", conf + " (1 graine)",
                "imagerie, a risque", N["A_corr"], m, s, "14")

if sources["Membre A — stabilite CNN"]:
    st = pd.read_csv(A / "3_stabilite_CNN" / "resume_graines.csv")
    for _, r in st.iterrows():
        ajouter("RQ1", "Fusion vs unimodal (deep learning)",
                f"CNN {r['configuration']} — {r['methode']} (5 graines)",
                "imagerie, a risque", N["A_corr"], r["mean"], r["std"], "14")

# ---------- RQ2 : le longitudinal apporte-t-il quelque chose ? ----------
d = sources["B — 12 longitudinal"]
if d is not None:
    t = pd.read_csv(d / "RQ2_statique_vs_longitudinal.csv", index_col=0)
    for modele in t.index:
        m, s = nombre(t.loc[modele, "ROC-AUC"])
        ajouter("RQ2", "Longitudinal vs statique", modele, "359 (>=4 visites)", 352, m, s, "12")

# ---------- RQ3 : robustesse a une donnee manquante ----------
if sources["Membre A — corrigee"]:
    t = pd.read_csv(CORR / "RQ3_robustesse.csv")
    for _, r in t.iterrows():
        ajouter("RQ3", "Modalite manquante", r["situation"], "imagerie, a risque",
                N["A_corr"], r["AUC"], r.get("AUC_ecart_type"), "10")

d = sources["B — 12 longitudinal"]
if d is not None:
    t = pd.read_csv(d / "RQ3_visite_manquante.csv")
    for _, r in t.iterrows():
        m, s = nombre(r["ROC-AUC"])
        ajouter("RQ3", "Visite manquante", r["situation"], "359 (>=4 visites)", 352, m, s, "12")

# ---------- Choix du modele tabulaire de reference ----------
d = sources["B — 11b modeles (ancree)"]
if d is not None:
    t = pd.read_csv(d / "classement_AUC_ancree.csv")
    for _, r in t.iterrows():
        if r["modele"] != "Hasard (reference)":
            ajouter("Modele", "Comparaison d'algorithmes (tabulaire)", r["modele"],
                    "359 ancree, a risque", N["B_ancree"],
                    r["roc_auc_moyenne"], r["roc_auc_ecart_type"], "11b")

synthese = pd.DataFrame(lignes)
print("Lignes collectees :", len(synthese))
display(synthese)

In [ ]:
# --- Enregistrement du tableau de synthese ---
SORTIE = C.sortie("synthese_finale")
synthese.to_csv(SORTIE / "synthese_tous_resultats.csv", index=False, encoding="utf-8-sig")

# Tableau resume : le meilleur de chaque groupe, pour les diapos.
resume = []
for rq, question, groupe in [
    ("RQ1", "Tabulaire seul", ["Tabulaire seul"]),
    ("RQ1", "Imagerie seule (meilleure)", ["Imagerie IRM seule", "Imagerie PET seule", "Imagerie IRM+PET"]),
    ("RQ1", "Fusion", ["Fusion (tabulaire + imagerie)"]),
]:
    sous = synthese.loc[(synthese["RQ"] == "RQ1") & synthese["Modele / configuration"].isin(groupe)]
    if len(sous):
        meilleur = sous.loc[sous["ROC-AUC"].idxmax()]
        resume.append({"RQ": rq, "Comparaison": question,
                       "Modele": meilleur["Modele / configuration"],
                       "ROC-AUC": meilleur["ROC-AUC"], "ecart-type": meilleur["ecart-type"],
                       "N": meilleur["N"]})

for etiquette, prefixe in [("Statique (meilleur)", "Statique"), ("Longitudinal (meilleur)", "Longitudinal")]:
    sous = synthese.loc[(synthese["RQ"] == "RQ2")
                        & synthese["Modele / configuration"].str.startswith(prefixe)]
    if len(sous):
        meilleur = sous.loc[sous["ROC-AUC"].idxmax()]
        resume.append({"RQ": "RQ2", "Comparaison": etiquette,
                       "Modele": meilleur["Modele / configuration"],
                       "ROC-AUC": meilleur["ROC-AUC"], "ecart-type": meilleur["ecart-type"],
                       "N": meilleur["N"]})

resume = pd.DataFrame(resume)
print("=== Resume pour la presentation ===")
display(resume)
resume.to_csv(SORTIE / "resume_presentation.csv", index=False, encoding="utf-8-sig")
print("Enregistre dans :", SORTIE)

In [ ]:
# --- Figure de synthese : un panneau par question de recherche ---
groupes = [("RQ1", "RQ1 — fusion multimodale (imagerie)"),
           ("RQ2", "RQ2 — longitudinal vs statique"),
           ("RQ3", "RQ3 — donnee manquante"),
           ("Modele", "Choix du modele tabulaire")]
presents = [(rq, t) for rq, t in groupes if (synthese["RQ"] == rq).any()]

fig, axes = plt.subplots(len(presents), 1, constrained_layout=True,
                         figsize=(9, sum(0.32 * (synthese["RQ"] == rq).sum() + 1.2
                                         for rq, _ in presents)))
if len(presents) == 1:
    axes = [axes]

for ax, (rq, titre) in zip(axes, presents):
    sous = synthese.loc[synthese["RQ"] == rq].sort_values("ROC-AUC")
    pos = np.arange(len(sous))
    ect = sous["ecart-type"].fillna(0).to_numpy()
    ax.barh(pos, sous["ROC-AUC"], xerr=ect, color="#4C72B0", capsize=3)
    ax.set_yticks(pos)
    ax.set_yticklabels([f"{m}  (N={n})" for m, n in
                        zip(sous["Modele / configuration"], sous["N"])], fontsize=7)
    ax.axvline(0.5, color="grey", ls="--", lw=1)
    ax.set_xlim(0.35, 1.0)
    for p, v, e in zip(pos, sous["ROC-AUC"], ect):
        ax.text(v + e + 0.012, p, f"{v:.3f}", va="center", fontsize=7)
    ax.set_title(titre, fontsize=10)
    ax.set_xlabel("ROC-AUC")

fig.savefig(SORTIE / "synthese_figure.png", dpi=140)
plt.show()
print("Figure :", SORTIE / "synthese_figure.png")

In [ ]:
# --- Conclusions automatiques par question de recherche ---
SEUIL_BRUIT = 0.05   # en dessous, l'ecart n'est pas distinguable du bruit

def meilleur_de(rq, filtre):
    sous = synthese.loc[(synthese["RQ"] == rq) & filtre(synthese)]
    return None if sous.empty else sous.loc[sous["ROC-AUC"].idxmax()]

print("=" * 78)
tab1 = meilleur_de("RQ1", lambda d: d["Modele / configuration"].eq("Tabulaire seul"))
fus = meilleur_de("RQ1", lambda d: d["Modele / configuration"].str.startswith("Fusion"))
img = meilleur_de("RQ1", lambda d: d["Modele / configuration"].str.startswith("Imagerie"))
if tab1 is not None and fus is not None:
    gain = fus["ROC-AUC"] - tab1["ROC-AUC"]
    verdict = "PAS de gain mesurable" if abs(gain) < SEUIL_BRUIT else ("gain" if gain > 0 else "perte")
    print(f"RQ1 (N={tab1['N']}) : tabulaire {tab1['ROC-AUC']:.3f} | "
          f"imagerie {img['ROC-AUC']:.3f} | fusion {fus['ROC-AUC']:.3f}")
    print(f"     -> ecart fusion - tabulaire = {gain:+.3f} : {verdict} "
          f"(seuil de bruit {SEUIL_BRUIT})")

sta = meilleur_de("RQ2", lambda d: d["Modele / configuration"].str.startswith("Statique"))
lon = meilleur_de("RQ2", lambda d: d["Modele / configuration"].str.startswith("Longitudinal"))
if sta is not None and lon is not None:
    gain = lon["ROC-AUC"] - sta["ROC-AUC"]
    print(f"\nRQ2 (N={sta['N']}) : statique {sta['ROC-AUC']:.3f} | "
          f"longitudinal {lon['ROC-AUC']:.3f}  -> {gain:+.3f}")

rq3 = synthese.loc[synthese["RQ"] == "RQ3"]
if len(rq3):
    print("\nRQ3 :")
    for _, r in rq3.iterrows():
        print(f"     {r['Question']:20s} | {r['Modele / configuration']:45s} {r['ROC-AUC']:.3f}")

mod = synthese.loc[synthese["RQ"] == "Modele"]
if len(mod):
    best = mod.loc[mod["ROC-AUC"].idxmax()]
    print(f"\nModele tabulaire retenu (N={best['N']}) : {best['Modele / configuration']} "
          f"= {best['ROC-AUC']:.3f} ± {best['ecart-type']}")
print("=" * 78)
print("\nRappel : ne comparez jamais deux lignes de cohortes differentes.")